# Frontal Ramping Analysis
**Steven Errington, 2026**

Analysis of foreperiod ramping activity across frontal cortical areas in macaque. Pipeline covers:
1. Setup and directory configuration
2. Spike log loading
3. SDF extraction aligned to stimulus onset, split by foreperiod duration
4. Z-scoring and linear ramp detection (circular-shift permutation test)
5. Prevalence analysis across areas (chi-square, odds ratios, BH-FDR)
6. Figures: stacked prevalence bars + forest plot; area-averaged SDFs by foreperiod

## 0. Imports and configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

from ramping_utils import smooth, load_sdf_mat, load_beh_mat
# If your .mat files were saved with -v7.3 (HDF5), import this instead:
# from ramping_utils import load_sdf_mat_hdf5 as load_sdf_mat

warnings.filterwarnings('ignore', category=RuntimeWarning)

# ── Seaborn / matplotlib style ──────────────────────────────────────────────
sns.set_theme(style='ticks', font_scale=0.9)
plt.rcParams.update({
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        150,
})

## 1. Directory setup

In [ ]:
# ── Edit these paths to match your local setup ──────────────────────────────
dirs = {
    'root':      Path('/Volumes/Mnemosyne/Codespace/2026-frontal-ramping'),
    'raw_data':  Path('/Volumes/Mnemosyne/Data/2026_macaque_value/spk/'),
    'proc_data': Path('/Volumes/Mnemosyne/Data/2026_macaque_value/proc/'),
    'figures':   Path('/Volumes/Mnemosyne/Codespace/2026-frontal-ramping/_figures/'),
}

data_dir = dirs['root'] / '_data'
data_dir.mkdir(parents=True, exist_ok=True)
dirs['figures'].mkdir(parents=True, exist_ok=True)

## 2. Load spike log

Reads the pre-built `spike_log.csv` (equivalent to `get_spk_map` output). Each row is one neuron; columns include `session`, `neuron_label`, `area`.

In [ ]:
spike_log = pd.read_csv(data_dir / 'spike_log.csv')
n_neurons = len(spike_log)

print(f'Loaded spike_log: {n_neurons} neurons across {spike_log["area"].nunique()} areas')
spike_log.head()

## 3. I/O helpers

`smooth`, `load_sdf_mat`, and `load_beh_mat` are imported from `ramping_utils.py`.
That module also provides `load_sdf_mat_hdf5` as a drop-in replacement if your
`.mat` files were saved with MATLAB's `-v7.3` flag — swap the import in cell 0 if needed.

In [ ]:
# Confirm imports from ramping_utils resolved correctly
print(f'smooth      : {smooth}')
print(f'load_sdf_mat: {load_sdf_mat}')
print(f'load_beh_mat: {load_beh_mat}')

## 4. Build `fixation_array`

Shape: `(n_neurons, n_timepoints, 4)` where dim 3 indexes foreperiod condition:
- 0 → all foreperiods (>500 ms)
- 1 → short (700–750 ms)
- 2 → medium (900–1100 ms)
- 3 → long (>1200 ms)

Each entry is a trial-averaged, smoothed SDF aligned to stimulus onset.

**Note:** the smoothing window (100 samples) matches MATLAB's `smooth(..., 100)`.

In [ ]:
SAVE_PATH = data_dir / 'fixation_array.npy'
SMOOTH_WIN = 100

if SAVE_PATH.exists():
    print('Loading existing fixation_array from disk...')
    fixation_array = np.load(SAVE_PATH)

else:
    # Determine n_timepoints from first valid file
    _first_sdf = load_sdf_mat(
        dirs['proc_data'] / f"{spike_log['neuron_label'].iloc[0]}.mat"
    )
    n_timepoints = _first_sdf['sdf']['stim_on'].shape[1]

    fixation_array = np.full((n_neurons, n_timepoints, 4), np.nan)

    print(f'Computing fixation_array for {n_neurons} neurons...')
    for neuron_i, row in tqdm(spike_log.iterrows(), total=n_neurons):

        beh_path = dirs['raw_data'] / f"{row['session']}_spk.mat"
        sdf_path = dirs['proc_data'] / f"{row['neuron_label']}.mat"

        try:
            beh = load_beh_mat(beh_path)
            sdf = load_sdf_mat(sdf_path)['sdf']['stim_on']  # (n_trials, n_time)
        except Exception as e:
            print(f'  [WARN] neuron {neuron_i}: {e}')
            continue

        foreperiod = beh['stim_on'] - beh['fixcross_fix']

        # Boolean index arrays (matching MATLAB logical indexing)
        masks = {
            'all':    foreperiod > 0.5,
            'short':  (foreperiod > 0.70) & (foreperiod < 0.75),
            'medium': (foreperiod > 0.90) & (foreperiod < 1.10),
            'long':   foreperiod > 1.20,
        }

        for cond_i, mask_key in enumerate(masks):
            mask = masks[mask_key]
            if mask.sum() == 0:
                continue
            trial_mean = np.nanmean(sdf[mask, :], axis=0)
            fixation_array[neuron_i, :, cond_i] = smooth(trial_mean, SMOOTH_WIN)

    np.save(SAVE_PATH, fixation_array)
    print(f'fixation_array saved to {SAVE_PATH}')

print(f'fixation_array shape: {fixation_array.shape}  (neurons x time x conditions)')

## 5. Z-score SDFs

In [ ]:
# Z-score each neuron x condition trace independently
# Equivalent to MATLAB zscore() applied per row-slice
z_fixation_array = np.zeros_like(fixation_array)

for neuron_i in range(n_neurons):
    for cond_i in range(fixation_array.shape[2]):
        trace = fixation_array[neuron_i, :, cond_i]
        mu, sd = np.nanmean(trace), np.nanstd(trace, ddof=1)
        if sd > 0:
            z_fixation_array[neuron_i, :, cond_i] = (trace - mu) / sd

print('Z-scoring complete.')

## 6. Ramp detection — linear fit + circular-shift permutation

For each neuron:
- Fit a line to the z-scored SDF in the 750 ms window before stimulus onset
- Compute R² of the fit
- Build a null R² distribution by circularly shifting the trace 1000 times
- p-value = proportion of null R² ≥ observed R²
- Threshold: p < 0.001

In [ ]:
SDF_ZERO  = 2000          # index of t=0 (stim_on) in the 4001-sample SDF
RAMP_MS   = np.arange(-750, 1)  # −750 to 0 ms
RAMP_IDX  = SDF_ZERO + RAMP_MS  # absolute indices into SDF
N_PERM    = 1000
N_PTS     = len(RAMP_MS)
t         = RAMP_MS.astype(float)


def compute_rsq(y: np.ndarray, t: np.ndarray) -> float:
    """R² of a linear fit to y ~ t."""
    coeffs  = np.polyfit(t, y, 1)
    y_hat   = np.polyval(coeffs, t)
    ss_res  = np.sum((y - y_hat) ** 2)
    ss_tot  = np.sum((y - y.mean()) ** 2)
    if ss_tot == 0:
        return 0.0
    return 1.0 - ss_res / ss_tot


LINEARITY_PATH = data_dir / 'linearity_analysis.csv'

if LINEARITY_PATH.exists():
    print('Loading existing linearity_analysis.csv...')
    linearity_df = pd.read_csv(LINEARITY_PATH)

else:
    rng = np.random.default_rng(42)  # reproducible permutations

    records = []
    print(f'Running ramp detection for {n_neurons} neurons ({N_PERM} permutations each)...')

    for neuron_i in tqdm(range(n_neurons)):
        fr = z_fixation_array[neuron_i, RAMP_IDX, 0]  # all-FP z-scored SDF

        if np.all(np.isnan(fr)):
            records.append({'flag': 0, 'pval': np.nan, 'slope': np.nan, 'r2': np.nan})
            continue

        # Observed slope + R²
        coeffs = np.polyfit(t, fr, 1)
        slope  = coeffs[0]
        rsq    = compute_rsq(fr, t)

        # Null distribution via circular shifts
        rsq_null = np.empty(N_PERM)
        for perm_i in range(N_PERM):
            shift_amt       = rng.integers(1, N_PTS)
            fr_shift        = np.roll(fr, shift_amt)
            rsq_null[perm_i] = compute_rsq(fr_shift, t)

        p_val      = np.mean(rsq_null >= rsq)
        is_ramping = int(p_val < 0.001)

        records.append({'flag': is_ramping, 'pval': p_val, 'slope': slope, 'r2': rsq})

    linearity_df = pd.DataFrame(records)
    linearity_df['area']         = spike_log['area'].values
    linearity_df['neuron_label'] = spike_log['neuron_label'].values

    linearity_df.to_csv(LINEARITY_PATH, index=False)
    print(f'linearity_analysis.csv saved to {LINEARITY_PATH}')

linearity_df.head()

## 7. Classify ramping neurons

In [ ]:
R2_THRESH = 0.8

pos_mask = (linearity_df['flag'] == 1) & (linearity_df['slope'] > 0) & (linearity_df['r2'] > R2_THRESH)
neg_mask = (linearity_df['flag'] == 1) & (linearity_df['slope'] < 0) & (linearity_df['r2'] > R2_THRESH)

print(f'Positive ramp neurons: {pos_mask.sum()}')
print(f'Negative ramp neurons: {neg_mask.sum()}')

# Build analysis_log (spike_log + ramp classification)
analysis_log = spike_log.copy()
analysis_log['ramping_fp_flag'] = 0
analysis_log.loc[pos_mask, 'ramping_fp_flag'] =  1
analysis_log.loc[neg_mask, 'ramping_fp_flag'] = -1

# Convenience index dicts (matching MATLAB neuron_idx struct)
neuron_idx = {
    'pos_ramping': np.where(pos_mask)[0],
    'neg_ramping': np.where(neg_mask)[0],
}

## 8. Prevalence per area

In [ ]:
areas  = sorted(analysis_log['area'].unique())

rows = []
for area in areas:
    flags = analysis_log.loc[analysis_log['area'] == area, 'ramping_fp_flag']
    n_total   = len(flags)
    n_pos     = (flags ==  1).sum()
    n_neg     = (flags == -1).sum()
    n_nonramp = (flags ==  0).sum()
    n_ramp    = n_pos + n_neg
    rows.append({
        'area': area, 'n_total': n_total,
        'n_pos': n_pos, 'n_neg': n_neg,
        'n_nonramp': n_nonramp, 'n_ramp': n_ramp,
        'pct_ramp': 100 * n_ramp / n_total if n_total > 0 else np.nan,
    })

prevalence_df = pd.DataFrame(rows).sort_values('pct_ramp', ascending=False)
prevalence_df

## 9. Reorder by cytoarchitectonic hierarchy and apply minimum-n filter

In [ ]:
# Cytoarchitectonic order (matches MATLAB area_order)
AREA_ORDER = [
    '24c',
    '6DR', '6DC', '6VaVb',
    '8B', '8A', '46d', '46df', '46v',
    '44', '45',
    '12r', '12m', '12o', '12l',
    'AI',
    '13l', '13m', '11ml',
    'cd', 'pu',
    'AMG',
]

# Region membership and colours
REGION_MAP_FULL = {
    '24c':   ('MFC',      '#993399'),
    '6DR':   ('PMC',      '#E67F14'), '6DC': ('PMC', '#E67F14'), '6VaVb': ('PMC', '#E67F14'),
    '8B':    ('dlPFC',    '#339EE5'), '8A': ('dlPFC', '#339EE5'), '46d': ('dlPFC', '#339EE5'),
    '46df':  ('dlPFC',    '#339EE5'), '46v': ('dlPFC', '#339EE5'),
    '44':    ('IFG',      '#4DB366'), '45': ('IFG', '#4DB366'),
    '12r':   ('vlPFC',    '#1A66B3'), '12m': ('vlPFC', '#1A66B3'),
    '12o':   ('vlPFC',    '#1A66B3'), '12l': ('vlPFC', '#1A66B3'),
    'AI':    ('AI',       '#B3B34C'),
    '13l':   ('OFC',      '#CC4C4C'), '13m': ('OFC', '#CC4C4C'), '11ml': ('OFC', '#CC4C4C'),
    'cd':    ('Striatum', '#808080'), 'pu': ('Striatum', '#808080'),
    'AMG':   ('AMG',      '#4C4C4C'),
}

MIN_N = 10

# Filter and reorder
pt = prevalence_df[prevalence_df['n_total'] >= MIN_N].copy()
order_present = [a for a in AREA_ORDER if a in pt['area'].values]
pt = pt.set_index('area').loc[order_present].reset_index()

# Attach region labels and colours
pt['region'] = pt['area'].map(lambda a: REGION_MAP_FULL.get(a, ('Unknown', '#AAAAAA'))[0])
pt['color']  = pt['area'].map(lambda a: REGION_MAP_FULL.get(a, ('Unknown', '#AAAAAA'))[1])

n_plot = len(pt)
print(f'{n_plot} areas pass minimum-n threshold of {MIN_N}')
pt[['area', 'region', 'n_total', 'n_ramp', 'pct_ramp']]

## 10. Statistics: omnibus chi-square + per-area odds ratios with BH-FDR

In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────

def chi2_contingency_manual(cont: np.ndarray):
    """Chi-square test on an arbitrary 2D contingency table (no toolbox)."""
    cont  = cont.astype(float)
    row_s = cont.sum(axis=1, keepdims=True)
    col_s = cont.sum(axis=0, keepdims=True)
    N     = cont.sum()
    exp   = (row_s @ col_s) / N
    chi2  = np.sum((cont - exp) ** 2 / exp)
    df    = (cont.shape[0] - 1) * (cont.shape[1] - 1)
    p     = 1 - stats.chi2.cdf(chi2, df)
    return chi2, p, df


def bh_fdr(p_vals: np.ndarray) -> np.ndarray:
    """Benjamini-Hochberg FDR correction."""
    n = len(p_vals)
    sort_idx    = np.argsort(p_vals)
    p_sorted    = p_vals[sort_idx]
    p_fdr_sort  = p_sorted * n / (np.arange(n) + 1)
    # Enforce monotonicity (cumulative min from the right)
    for k in range(n - 2, -1, -1):
        p_fdr_sort[k] = min(p_fdr_sort[k], p_fdr_sort[k + 1])
    p_fdr = np.empty(n)
    p_fdr[sort_idx] = np.minimum(p_fdr_sort, 1.0)
    return p_fdr


def odds_ratio_woolf(a, b, c, d):
    """
    Odds ratio and 95% CI (Woolf logit method).
    Haldane–Anscombe correction applied when any cell is zero.
    """
    if 0 in (a, b, c, d):
        a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    log_or    = np.log((a * d) / (b * c))
    se_log_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
    or_val    = np.exp(log_or)
    ci_lo     = np.exp(log_or - 1.96 * se_log_or)
    ci_hi     = np.exp(log_or + 1.96 * se_log_or)
    return or_val, ci_lo, ci_hi


# ── Omnibus tests ────────────────────────────────────────────────────────────

cont_ramp  = pt[['n_ramp', 'n_nonramp']].values
chi2_omni_ramp, p_omni_ramp, df_omni_ramp = chi2_contingency_manual(cont_ramp)
print(f'Omnibus ramp vs non-ramp: X²({df_omni_ramp}) = {chi2_omni_ramp:.2f}, p = {p_omni_ramp:.4f}')

ramp_rows  = pt['n_ramp'] > 0
cont_posneg = pt.loc[ramp_rows, ['n_pos', 'n_neg']].values
chi2_omni_pn, p_omni_pn, df_omni_pn = chi2_contingency_manual(cont_posneg)
print(f'Omnibus pos vs neg:       X²({df_omni_pn}) = {chi2_omni_pn:.2f}, p = {p_omni_pn:.4f}')


# ── Per-area OR (one-vs-rest) ────────────────────────────────────────────────

overall_ramp    = pt['n_ramp'].sum()
overall_total   = pt['n_total'].sum()
overall_nonramp = overall_total - overall_ramp

chi2_stat_list, chi2_p_list = [], []
or_list, ci_lo_list, ci_hi_list = [], [], []

for _, row in pt.iterrows():
    a = row['n_ramp']
    b = row['n_nonramp']
    c = overall_ramp    - a
    d = overall_nonramp - b

    # Chi-square with Yates correction (2×2)
    cont    = np.array([[a, b], [c, d]], dtype=float)
    row_s   = cont.sum(axis=1, keepdims=True)
    col_s   = cont.sum(axis=0, keepdims=True)
    N_      = cont.sum()
    exp_    = (row_s @ col_s) / N_
    yates   = np.sum((np.abs(cont - exp_) - 0.5) ** 2 / exp_)
    p_yates = 1 - stats.chi2.cdf(yates, 1)

    if (exp_ < 5).any():
        warnings.warn(f"Area {row['area']}: expected cell < 5")

    chi2_stat_list.append(yates)
    chi2_p_list.append(p_yates)

    or_val, ci_lo, ci_hi = odds_ratio_woolf(a, b, c, d)
    or_list.append(or_val)
    ci_lo_list.append(ci_lo)
    ci_hi_list.append(ci_hi)

chi2_p_arr   = np.array(chi2_p_list)
chi2_p_fdr   = bh_fdr(chi2_p_arr)

pt = pt.copy()
pt['chi2_stat']  = chi2_stat_list
pt['chi2_p_raw'] = chi2_p_arr
pt['chi2_p_fdr'] = chi2_p_fdr
pt['sig_fdr']    = chi2_p_fdr < 0.05
pt['odds_ratio'] = or_list
pt['or_ci_lo']   = ci_lo_list
pt['or_ci_hi']   = ci_hi_list

print('\nPer-area results (FDR corrected):')
display(pt[['area', 'n_total', 'n_ramp', 'pct_ramp',
            'odds_ratio', 'or_ci_lo', 'or_ci_hi',
            'chi2_p_raw', 'chi2_p_fdr', 'sig_fdr']].round(3))

## 11. Figure 1: Prevalence bars + forest plot

Reproduces the three-panel MATLAB figure from `inspect_foreperiod_neurons.m`:
- **Upper panel**: stacked bars — positive ramp (up from zero) and negative ramp (down)
- **Lower panel**: odds ratio forest plot per area (log scale), FDR significance markers

In [ ]:
def sig_marker(p: float) -> str:
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return ''


def add_region_dividers(ax, pt: pd.DataFrame):
    """Dashed vertical lines at boundaries between cytoarchitectonic regions."""
    regions = pt['region'].values
    bounds  = np.where(np.array(regions[:-1]) != np.array(regions[1:]))[0]
    for b in bounds:
        ax.axvline(b + 0.5, color='#BBBBBB', linewidth=0.8, linestyle='--')


pos_pct = 100 * pt['n_pos'] / pt['n_total']
neg_pct = 100 * pt['n_neg'] / pt['n_total']
x       = np.arange(n_plot)
colors  = pt['color'].values

fig, axes = plt.subplots(
    2, 1, figsize=(12, 8),
    gridspec_kw={'height_ratios': [2, 1]},
    constrained_layout=True
)

# ── Panel 1: Stacked prevalence bars ────────────────────────────────────────
ax = axes[0]

for i, (xi, pos, neg, col) in enumerate(zip(x, pos_pct, neg_pct, colors)):
    ax.bar(xi, pos,  color=col,           width=0.7, linewidth=0)
    ax.bar(xi, -neg, color=col, alpha=0.5, width=0.7, linewidth=0)

ax.axhline(0, color='k', linewidth=0.8)
add_region_dividers(ax, pt)

ax.set_xlim(-0.5, n_plot - 0.5)
ax.set_ylim(-30, 30)
ax.set_xticks(x)
ax.set_xticklabels([])
ax.set_yticks(range(-30, 31, 10))
ax.set_yticklabels([str(abs(v)) for v in range(-30, 31, 10)])
ax.set_ylabel('Ramp prevalence (%)')
ax.text(-1.2, 15, 'Positive', rotation=90, ha='center', va='center', fontsize=8)
ax.text(-1.2, -15, 'Negative', rotation=90, ha='center', va='center', fontsize=8)

title_str = (
    f'Omnibus ramp vs non-ramp: X²({df_omni_ramp})={chi2_omni_ramp:.1f}, p={p_omni_ramp:.4f}\n'
    f'Omnibus pos vs neg: X²({df_omni_pn})={chi2_omni_pn:.1f}, p={p_omni_pn:.4f}'
)
ax.set_title(title_str, fontsize=8, loc='left')
sns.despine(ax=ax, bottom=True)


# ── Panel 2: Forest plot ─────────────────────────────────────────────────────
ax2 = axes[1]

ax2.axhline(1, color='k', linewidth=0.8, linestyle='--')
add_region_dividers(ax2, pt)

for i, row in pt.reset_index(drop=True).iterrows():
    col   = row['color']
    or_   = row['odds_ratio']
    ci_lo = row['or_ci_lo']
    ci_hi = row['or_ci_hi']

    ax2.plot([i, i], [ci_lo, ci_hi], color=col, linewidth=1.5)

    if row['sig_fdr']:
        ax2.scatter(i, or_, s=60, color=col, zorder=5)
    else:
        ax2.scatter(i, or_, s=60, facecolors='none', edgecolors=col,
                    linewidths=1.2, zorder=5)

    marker = sig_marker(row['chi2_p_fdr'])
    if marker:
        ax2.text(i, 12, marker, ha='center', va='center', fontsize=8)

    ax2.text(i, 0.12, f"n={int(row['n_total'])}",
             ha='center', va='center', fontsize=6)

ax2.set_yscale('log')
ax2.set_ylim(0.05, 20)
ax2.set_xlim(-0.5, n_plot - 0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(pt['area'], rotation=45, ha='right', fontsize=9)
ax2.set_ylabel('Odds ratio vs rest\n(log scale)')
sns.despine(ax=ax2)

fig.savefig(dirs['figures'] / 'foreperiod_prevalence_and_OR.pdf', bbox_inches='tight')
plt.show()

## 12. Figure 2: Area-averaged SDFs by foreperiod condition

Reproduces `untitled4.m`: one figure per ramp type (positive / negative), one subplot per area, showing mean ± SEM traces for short/medium/long foreperiod conditions.

In [ ]:
TIME_VEC  = np.arange(-2000, 2001)
PLOT_MASK = (TIME_VEC >= -1500) & (TIME_VEC <= 250)
TIME_PLOT = TIME_VEC[PLOT_MASK]
FP_TIMES  = [-1300, -1000, -700]   # foreperiod onset lines

# Conditions 1–3 (0-indexed) = short, medium, long
COND_LABELS = ['Short', 'Medium', 'Long']
COND_COLORS = ['#4CA5E8', '#1A66B3', '#00173F']   # light → dark blue

N_COLS = 4

RAMP_TYPES = [
    {'label': 'Positive ramp', 'flag':  1},
    {'label': 'Negative ramp', 'flag': -1},
]

for ramp_type in RAMP_TYPES:
    flag  = ramp_type['flag']
    label = ramp_type['label']

    areas_plot = pt['area'].values
    n_areas    = len(areas_plot)
    n_rows     = int(np.ceil(n_areas / N_COLS))

    fig, axes = plt.subplots(
        n_rows, N_COLS,
        figsize=(N_COLS * 3.5, n_rows * 2.8),
        constrained_layout=True
    )
    axes_flat = axes.flatten()
    fig.suptitle(f'Foreperiod SDF — {label} neurons', fontsize=13)

    for area_i, area in enumerate(areas_plot):
        ax = axes_flat[area_i]

        area_mask = (
            (analysis_log['area'] == area) &
            (analysis_log['ramping_fp_flag'] == flag)
        )
        neuron_indices = np.where(area_mask)[0]
        n_area = len(neuron_indices)

        ax.set_title(f'{area}  (n={n_area})', fontsize=9)
        ax.set_xlim(-1500, 250)
        ax.set_xlabel('Time from stim on (ms)', fontsize=7)
        ax.set_ylabel('FR (z)', fontsize=7)
        ax.tick_params(labelsize=7)
        sns.despine(ax=ax)

        if n_area == 0:
            ax.text(np.mean([-1500, 250]), 0, 'No neurons',
                    ha='center', fontsize=8, color='#AAAAAA')
            continue

        # z_fixation_array: (n_neurons, n_time, 4); conds 1-3 = short/med/long
        area_data = z_fixation_array[np.ix_(neuron_indices, np.where(PLOT_MASK)[0])]
        # shape: (n_area, n_time_plot, 4)
        # Actually need all 4 dims, so use full indexing
        area_data = z_fixation_array[neuron_indices, :, :][:, PLOT_MASK, :]  # (n_area, n_time, 4)

        for cond_i, (clabel, ccolor) in enumerate(zip(COND_LABELS, COND_COLORS)):
            cond_data = area_data[:, :, cond_i + 1]  # +1: skip 'all' at index 0

            mu  = np.nanmean(cond_data, axis=0)
            sem = np.nanstd(cond_data, axis=0, ddof=1) / np.sqrt(n_area)

            ax.fill_between(TIME_PLOT, mu - sem, mu + sem,
                            color=ccolor, alpha=0.15)
            ax.plot(TIME_PLOT, mu, color=ccolor, linewidth=1.5,
                    label=clabel)

        for fp in FP_TIMES:
            ax.axvline(fp, color='#BBBBBB', linewidth=0.8, linestyle='--')
        ax.axvline(0, color='#333333', linewidth=1.0)

        if area_i == 0:
            ax.legend(fontsize=7, loc='upper left', frameon=False)

    # Hide any unused subplots
    for ax in axes_flat[n_areas:]:
        ax.set_visible(False)

    fname = f"foreperiod_sdf_{'pos' if flag == 1 else 'neg'}_ramp.pdf"
    fig.savefig(dirs['figures'] / fname, bbox_inches='tight')
    plt.show()

## 13. Figure 3: Example single-unit raster + SDF

Reproduces `untitled3.m`: one positive and one negative example neuron, shown with raster and foreperiod-split SDF traces side by side.

In [ ]:
# ── Select example neurons ───────────────────────────────────────────────────
# Edit these indices to match the neurons you want to highlight
POS_EXAMPLE_IDX = neuron_idx['pos_ramping'][35]   # 35th positive ramp neuron
NEG_EXAMPLE_IDX = neuron_idx['neg_ramping'][40]   # 40th negative (MATLAB used 403; adjust)

EXAMPLE_NEURONS = [POS_EXAMPLE_IDX, NEG_EXAMPLE_IDX]
EXAMPLE_LABELS  = ['Positive ramp', 'Negative ramp']
YLIM_VALS       = [(0, 20), (0, 30)]
XLIM            = (-1500, 250)
FP_COLORS       = ['#5BC3F0', '#1A66B3', '#002D6E']   # short / medium / long
FP_KEYS         = ['short', 'medium', 'long']
FP_LABELS       = ['Short', 'Medium', 'Long']

fig, axes = plt.subplots(
    2, 2, figsize=(10, 7),
    gridspec_kw={'height_ratios': [1, 2.5]},
    constrained_layout=True
)

for col, (neuron_i, ex_label, ylim) in enumerate(zip(EXAMPLE_NEURONS, EXAMPLE_LABELS, YLIM_VALS)):

    row_info    = spike_log.iloc[neuron_i]
    neuron_label = row_info['neuron_label']

    beh = load_beh_mat(dirs['raw_data'] / f"{row_info['session']}_spk.mat")
    sdf_data = load_sdf_mat(dirs['proc_data'] / f"{neuron_label}.mat")
    sdf_mat  = sdf_data['sdf']['stim_on'].astype(float)   # (n_trials, n_time)

    # Smooth each trial (matching MATLAB per-trial smooth before gramm)
    for ti in range(sdf_mat.shape[0]):
        sdf_mat[ti, :] = smooth(sdf_mat[ti, :], 100)

    foreperiod = beh['stim_on'] - beh['fixcross_fix']
    masks = {
        'short':  (foreperiod > 0.70) & (foreperiod < 0.75),
        'medium': (foreperiod > 0.90) & (foreperiod < 1.10),
        'long':   foreperiod > 1.20,
    }

    # ── Raster ────────────────────────────────────────────────────────────
    ax_raster = axes[0, col]
    trial_offset = 0

    for fp_key, fp_col, fp_lab in zip(FP_KEYS, FP_COLORS, FP_LABELS):
        trial_idx = np.where(masks[fp_key])[0]
        for t_i, trial in enumerate(trial_idx):
            # raster: spike times are in the sdf_data raster cell array
            # If rasters not loaded, skip raster panel and plot mean SDF only
            pass   # placeholder — raster data requires separate loading (see note below)
        trial_offset += len(trial_idx)

    ax_raster.set_xlim(XLIM)
    ax_raster.set_title(f'{ex_label}\n{neuron_label}', fontsize=9)
    ax_raster.set_ylabel('Trial', fontsize=8)
    ax_raster.tick_params(labelbottom=False)
    sns.despine(ax=ax_raster, bottom=True)

    # ── SDF ───────────────────────────────────────────────────────────────
    ax_sdf = axes[1, col]

    for fp_key, fp_col, fp_lab in zip(FP_KEYS, FP_COLORS, FP_LABELS):
        trial_mask = masks[fp_key]
        if trial_mask.sum() == 0:
            continue
        data_fp = sdf_mat[trial_mask, :][:, PLOT_MASK]
        mu      = np.nanmean(data_fp, axis=0)
        sem     = np.nanstd(data_fp, axis=0, ddof=1) / np.sqrt(trial_mask.sum())

        ax_sdf.fill_between(TIME_PLOT, mu - sem, mu + sem,
                            color=fp_col, alpha=0.15)
        ax_sdf.plot(TIME_PLOT, mu, color=fp_col, linewidth=1.8, label=fp_lab)

    for fp in FP_TIMES:
        ax_sdf.axvline(fp, color='#BBBBBB', linewidth=0.8, linestyle='--')
    ax_sdf.axvline(0, color='#333333', linewidth=1.0)

    ax_sdf.set_xlim(XLIM)
    ax_sdf.set_ylim(ylim)
    ax_sdf.set_xlabel('Time from stimulus onset (ms)', fontsize=8)
    ax_sdf.set_ylabel('Firing rate (sp/s)', fontsize=8)
    ax_sdf.legend(fontsize=7, frameon=False)
    sns.despine(ax=ax_sdf)

fig.savefig(dirs['figures'] / 'example_foreperiod_split_sdf.pdf', bbox_inches='tight')
plt.show()

print("""
Note: raster panels above are placeholders. Spike times are stored in sdf_data['raster']
but unpacking MATLAB cell arrays of variable-length spike times requires careful handling
of the squeeze_me output. If you want rasters populated, load with:
    mat = sio.loadmat(path, squeeze_me=True, struct_as_record=False)
    raster_stim = mat['raster'].stim_on   # object array of spike-time vectors
and iterate over raster_stim[trial_i] for each trial.
""")